This is the agent of DDQN

In [ ]:
import random
from collections import deque
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from src.rl.env import Array3D


import torch
import torch.nn as nn
import torch.nn.functional as F

class DuelingDQN(nn.Module):
    def __init__(self, input_channels=3, board_size=4, conv_filters=64):
        super(DuelingDQN, self).__init__()
        # Convolutional base: maintain spatial dimensions with padding
        self.conv1 = nn.Conv2d(input_channels, conv_filters, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(conv_filters)
        self.conv2 = nn.Conv2d(conv_filters, conv_filters, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(conv_filters)
        self.conv3 = nn.Conv2d(conv_filters, conv_filters, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(conv_filters)
        
        # The size remains board_size x board_size after conv layers (thanks to padding)
        self.feature_size = conv_filters * board_size * board_size
        
        # Value stream: estimates the overall value of the state
        self.fc_value = nn.Sequential(
            nn.Linear(self.feature_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
        # Advantage stream: estimates the advantage for each action (one per board cell)
        self.fc_advantage = nn.Sequential(
            nn.Linear(self.feature_size, 128),
            nn.ReLU(),
            nn.Linear(128, board_size * board_size)
        )
        self.board_size = board_size

    def forward(self, x):
        # x shape: (batch_size, 3, board_size, board_size)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        # Flatten the conv output
        x = x.view(x.size(0), -1)
        
        value = self.fc_value(x)  # shape: (batch_size, 1)
        advantage = self.fc_advantage(x)  # shape: (batch_size, board_size*board_size)
        
        # Reshape advantage to (batch_size, board_size*board_size)
        advantage = advantage.view(-1, self.board_size * self.board_size)
        # Combine streams: Q = V + (A - mean(A))
        q_vals = value + advantage - advantage.mean(dim=1, keepdim=True)
        # Reshape to spatial grid (if you prefer)
        q_vals = q_vals.view(-1, self.board_size, self.board_size)
        return q_vals




def _build_model():
    return nn.Sequential(
        nn.Conv2d(3, 512, kernel_size=3),
        nn.ReLU(),
        nn.ConvTranspose2d(512, 1, kernel_size=3),
    )

class DQNAgent:
    def __init__(self, n_qubits: int, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.997) -> None:
        self.model = DuelingDQN(input_channels=3, board_size=n_qubits)

        self.memory = deque(maxlen=20000)
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.learning_rate = 1e-4
        self.n_qubits = n_qubits
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    def replay(self, batch_size):
        minibatch = random.sample(self.memory, batch_size)
        all_current_q_values = []
        all_target_q_values = []
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                with torch.no_grad():
                    target = target + self.gamma * torch.max(self.model(torch.from_numpy(next_state[0]).float()))

            target_q_values = torch.zeros(size=(1, self.n_qubits, self.n_qubits))
            target_q_values[0, action[0], action[1]] = target


            current_q_values = torch.from_numpy(state[0]).float()

            all_target_q_values.append(target_q_values)
            all_current_q_values.append(current_q_values)
        target_q_values = torch.stack([item for item in all_target_q_values])
        current_q_values = self.model(torch.stack([item for item in all_current_q_values]))
        self.optimizer_step(current_q_values, target_q_values)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def remember(self, state: Tuple[Array3D, list, list],
                 action: Tuple[int, int],
                 reward: float,
                 next_state: Tuple[Array3D, list, list], done: bool):

        self.memory.append((state, action, reward, next_state, done))

    def act(self, state: Array3D, allowed_rows: list, allowed_cols: list) -> Tuple[int, int]:
        if np.random.rand() <= self.epsilon:
            row = random.choice(allowed_rows)
            col = random.choice(allowed_cols)
            return row, col
        q_values = self.model(torch.from_numpy(state).float()).cpu().detach()[0]
        q_values = q_values[allowed_rows][:, allowed_cols]
        row_idx, col_idx = divmod(torch.argmax(q_values).item(), q_values.size(1))

        selected_row = allowed_rows[row_idx]
        selected_col = allowed_cols[col_idx]

        return selected_row, selected_col

    def optimizer_step(self, state_action_values, expected_state_action_values):
        criterion = nn.HuberLoss()
        loss = criterion(expected_state_action_values, state_action_values)
        # Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        # In-place gradient clipping

        self.optimizer.step()
        torch.nn.utils.clip_grad_value_(self.model.parameters(), 100)


This is the environment of rl algorithm

In [ ]:
from typing import Tuple, Optional, Union, List, Any, Dict

import gym
import networkx as nx
import numpy as np
from gym.core import RenderFrame
from pauliopt.circuits import Circuit
from pauliopt.clifford.tableau import CliffordTableau
from pauliopt.clifford.tableau_synthesis import steiner_reduce_column
from pauliopt.gates import CX, H, S
from pauliopt.topologies import Topology
from pauliopt.utils import is_cutting

from src.utils import random_hscx_circuit, tableau_from_circuit

Array3D = np.array


class CliffordTableauEnv(gym.Env[Tuple[int, int], np.ndarray]):
    def __init__(self, n_qubits: int, nr_gates: int = 1000, topology: Topology = None):
        """
        Defines a RL environment, that describes the synthesis of a clifford tableau.

        :param n_qubits: Nr of qubits of the tableau
        :param nr_gates: Nr of gates of the tableau
        :param topology: Which topology to synth the tableau with
        """
        super(CliffordTableauEnv, self).__init__()
        self.n_qubits = n_qubits
        self.nr_gates = nr_gates
        self.clifford_tableau_to_reduce = None
        self.final_circuit = None
        self.qubits_reduced = 0
        if topology is None:
            self.topology = Topology.complete(self.n_qubits)
        else:
            self.topology = topology
        self.graph = self.topology.to_nx
        self.adjacency_matrix = nx.adjacency_matrix(self.graph).toarray()
        self.allowed_rows = list(range(self.n_qubits))
        self.allowed_cols = list(range(self.n_qubits))

    def reset(self, **kwargs):
        """

        :param kwargs:
        :return:
        """
        circuit = random_hscx_circuit(nr_qubits=self.n_qubits, nr_gates=self.nr_gates)
        clifford_tableau = CliffordTableau(self.n_qubits)
        clifford_tableau = tableau_from_circuit(clifford_tableau, circuit)
        self.clifford_tableau_to_reduce = clifford_tableau.inverse()
        self.final_circuit = Circuit(self.n_qubits)
        self.graph = self.topology.to_nx
        self.allowed_rows = list(range(self.n_qubits))
        self.allowed_cols = list(range(self.n_qubits))
        self.adjacency_matrix = nx.adjacency_matrix(self.graph).toarray()
        self.qubits_reduced = 0

        return self._get_obs(), self.allowed_rows, self.allowed_cols

    def get_current_stats(self) -> float:
        return self.final_circuit.to_qiskit().count_ops().get("cx", 0)

    def step(self, action: Tuple[int, int]) -> Tuple[Tuple[Array3D, list, list], float, bool, Dict[Any, Any]]:
        current_circuit = Circuit(self.n_qubits)

        def apply(gate_name: str, gate_data: tuple) -> None:
            if gate_name == "CNOT":
                self.clifford_tableau_to_reduce.append_cnot(gate_data[0], gate_data[1])
                self.final_circuit.add_gate(CX(gate_data[0], gate_data[1]))
                current_circuit.add_gate(CX(gate_data[0], gate_data[1]))
            elif gate_name == "H":
                self.clifford_tableau_to_reduce.append_h(gate_data[0])
                self.final_circuit.add_gate(H(gate_data[0]))
                current_circuit.add_gate(H(gate_data[0]))
            elif gate_name == "S":
                self.clifford_tableau_to_reduce.append_s(gate_data[0])
                self.final_circuit.add_gate(S(gate_data[0]))
                current_circuit.add_gate(S(gate_data[0]))
            else:
                raise Exception("Unknown Gate")

        pivot_row, pivot_col = action
        assert not is_cutting(pivot_col, self.graph)

        self.allowed_rows.remove(pivot_row)
        self.allowed_cols.remove(pivot_col)
        self.qubits_reduced += 1
        steiner_reduce_column(pivot_col, pivot_row, self.graph, self.clifford_tableau_to_reduce, apply)
        self.graph.remove_node(pivot_col)
        done = False
        if self.qubits_reduced >= self.n_qubits:
            final_permutation = np.argmax(self.clifford_tableau_to_reduce.x_matrix, axis=1)
            signs_copy_z = self.clifford_tableau_to_reduce.signs[
                           self.clifford_tableau_to_reduce.n_qubits: 2 * self.clifford_tableau_to_reduce.n_qubits].copy()

            for col in range(self.clifford_tableau_to_reduce.n_qubits):
                if signs_copy_z[col] != 0:
                    apply("H", (final_permutation[col],))
                    apply("S", (final_permutation[col],))
                    apply("S", (final_permutation[col],))
                    apply("H", (final_permutation[col],))

            for col in range(self.clifford_tableau_to_reduce.n_qubits):
                if self.clifford_tableau_to_reduce.signs[col] != 0:
                    apply("S", (final_permutation[col],))
                    apply("S", (final_permutation[col],))

            done = True

        reward = np.exp(-current_circuit.to_qiskit().count_ops().get("cx", 0))
        return (self._get_obs(), self.allowed_rows, self.allowed_cols), reward, done, {}

    def render(self) -> Optional[Union[RenderFrame, List[RenderFrame]]]:
        print(self.clifford_tableau_to_reduce)
        return None

    def _get_obs(self) -> Array3D:
        disallowed_rows = list(set(list(range(self.n_qubits))) - set(self.allowed_rows))
        disallowed_columns = list(set(list(range(self.n_qubits))) - set(self.allowed_cols))
        bitmap = np.ones((self.n_qubits, self.n_qubits))
        bitmap[disallowed_rows, :] = 0.0
        bitmap[:, disallowed_columns] = 0.0
        return np.stack([
            self.clifford_tableau_to_reduce.x_matrix,
            self.clifford_tableau_to_reduce.z_matrix,
            bitmap
        ], axis=0)

    def _was_selected_previously(self, pivot_row: int, pivot_column: int) -> bool:
        return pivot_row not in self.allowed_rows or pivot_column not in self.allowed_cols


Reinforcemnt learning main

In [ ]:
import matplotlib.pyplot as plt

from src.nn.brute_force_data import get_best_cnots
from src.rl.agent import DQNAgent
from src.rl.env import CliffordTableauEnv


def main(n_qubits=4, n_gates=100, n_episodes=1000, batch_size=2000):
    """Describes a DQN Algorithm to try to learn clifford tableau heuristics.L"""
    env = CliffordTableauEnv(n_qubits, n_gates)
    agent = DQNAgent(n_qubits)

    scores_episode = []

    env.reset()
    cnots, score = get_best_cnots(env.clifford_tableau_to_reduce, env.topology)[0]
    for episode in range(n_episodes):
        print(f"Episode: {episode}")
        state = env.reset()
        done = False
        while not done:
            action = agent.act(*state)

            next_state, reward, done, _ = env.step(action)
            if done:
                break

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if len(agent.memory) > batch_size:
                agent.replay(batch_size)

        if done:
            scores_episode.append(env.get_current_stats())
        else:
            scores_episode.append(-1)

    plt.plot(scores_episode, label="#CX over epochs")
    plt.axhline(y=score, color='red', linestyle='--', label="Best possible #CX")

    plt.xlabel("Epochs")
    plt.ylabel("#CX")
    plt.legend()
    plt.savefig("./dqn_agent.png")


if __name__ == '__main__':
    main()
